In [0]:
%load_ext autoreload
%autoreload 2

In [0]:
# Define source file path, target table name, and natural key
# table_name = "bronze_dev.hud.tbd"
# natural_key = ["tbd"]

base_url = "https://www.huduser.gov/hudapi/public"

In [0]:
token = dbutils.secrets.get(
    scope="api-secrets",
    key="hud-token"
).lstrip("\x00")
token

Set up `retrieve_hud_json`

In [0]:
import requests
import json
from http import HTTPStatus

def retrieve_hud_json(url:str, token):

    # Headers
    headers = {
        "Authorization": f"Bearer {token}"
    }

    # Request
    response = requests.get(url, headers=headers)

    # Check response
    if response.status_code != 200:
        status_desc = HTTPStatus(response.status_code).phrase
        print(f"Request failed with status code {response.status_code} \"{status_desc}\" for {url}")
        return None
    else:
        print(f"Successful for {url}")
    # Parse JSON
    data = response.json()
    
    return {'url':url, 'data':data}

Test Connecticut

In [0]:
url = f"{base_url}/fmr/listCounties/CT?updated=2025" #updated connecticut counties
print(url)
res = retrieve_hud_json(url, token)

if res is None:
    print('Failed to retrieve data or empty response')

res 

List of states

In [0]:
url = f"{base_url}/fmr/listStates" 
print(url)
states = retrieve_hud_json(url, token)
states_df = spark.createDataFrame(states["data"])
display(states_df)

Get list of state urls

In [0]:
from src.utils.helpers import fetch_all

list_of_state_urls = []

for row in states_df.collect():
    state_code = row["state_code"] 

    url = f"{base_url}/fmr/listCounties/{state_code}"
    if state_code == "CT":
        url += "?updated=2025"
    list_of_state_urls.append(url)

list_of_state_urls

For all state urls, fetch counties

In [0]:
# Retrieve data with rate limiting
counties = fetch_all(
    urls=list_of_state_urls[:1], # TEMPORARY
    token=token,
    request_fn=retrieve_hud_json
)
counties

In [0]:

counties_subset = counties[:1] # TEMPORARY
list_of_dicts = []
for batch_dict in counties_subset:
    for data_point in batch_dict["data"]:
        list_of_dicts.append(data_point)

counties_df = spark.createDataFrame(list_of_dicts)
display(counties_df)

In [0]:
from pyspark.sql import functions as F

counties_df_filtered = counties_df.filter(
    F.col("county_name").isNull() | (F.trim(F.col("county_name")) == "")
)

if counties_df_filtered.limit(1).count() > 0:
    print("Found records with null county_name")
    display(counties_df_filtered)

### Once initial frame of counties is set up, get individual information for each county

In [0]:
# Extract county codes from the counties dataframe
county_codes = [row.fips_code for row in counties_df.collect()]

# Build URLs for individual county data
county_detail_urls = [f"{base_url}/fmr/data/{code}" for code in county_codes]
county_detail_urls

In [0]:
# Retrieve detailed data for each county with rate limiting
county_details = fetch_all(
    urls=county_detail_urls[:1], # TEMPORARY 
    token=token,
    request_fn=retrieve_hud_json
)

In [0]:
county_details

In [0]:
from typing import List, Dict, Any

def normalize_hud_response(rows: List[Dict[str, Any]]):
    output = []

    for r in rows:
        payload = r["data"]["data"]

        county_name = payload.get("county_name")
        year = payload.get("year")
        url = r.get("url")

        basic = payload.get("basicdata")

        # CASE 1: list (zip-level / msa breakdown)
        if isinstance(basic, list):
            for row in basic:
                output.append({
                    "county_name": county_name,
                    "year": year,
                    "zip_code": row.get("zip_code"),
                    "efficiency": row.get("Efficiency"),
                    "one_bedroom": row.get("One-Bedroom"),
                    "two_bedroom": row.get("Two-Bedroom"),
                    "three_bedroom": row.get("Three-Bedroom"),
                    "four_bedroom": row.get("Four-Bedroom"),                    
                    "url": url,
                })

        # CASE 2: dict (county-level summary)
        elif isinstance(basic, dict):
            output.append({
                "county_name": county_name,
                "year": year,
                "zip_code": None,
                "efficiency": basic.get("Efficiency"),
                "one_bedroom": basic.get("One-Bedroom"),
                "two_bedroom": basic.get("Two-Bedroom"),
                "three_bedroom": basic.get("Three-Bedroom"),
                "four_bedroom": basic.get("Four-Bedroom"),
                "url": url,
            })

    return output


from pyspark.sql import Row

flat = normalize_hud_response(county_details)

df = spark.createDataFrame([Row(**r) for r in flat])
display(df)

In [0]:
# Define source file path, target table name, and natural key
# table_name = "bronze_dev.hud.tbd"
# natural_key = ["tbd"]

from pyspark.sql import SparkSession
import src.utils.helpers

df_prepped = src.utils.helpers.prep_bronze_api_df(df)
src.utils.helpers.upsert_table(
    table_name = "bronze_dev.hud.counties",
    df=df_prepped,
    natural_key = ["county_name","year","zip_code"],
    spark=spark
)
display(df_prepped)